#### Imports

In [1]:
import nflreadpy as nfl
import pandas as pd
import datetime as dt
import os
from dotenv import load_dotenv
from sqlalchemy import create_engine, text
from typing import List, Sequence, Optional

#### Load Weekly Data

In [2]:
weekly_stats = nfl.load_player_stats(seasons=[2017,2018,2019,2020, 2021, 2022,2023, 2024], summary_level="week")

In [3]:
weekly_stats = weekly_stats.to_pandas()

In [4]:
weekly_stats.head()

,player_id,player_name,player_display_name,position,position_group,headshot_url,season,week,season_type,team,...,pat_blocked,pat_pct,gwfg_made,gwfg_att,gwfg_missed,gwfg_blocked,gwfg_distance,fantasy_points,fantasy_points_ppr,game_id
0,00-0004091,P.Dawson,Phil Dawson,K,SPEC,https://static.www.nfl.com/image/private/f_aut...,2017,1,REG,ARI,...,0,1.0,0,0,0,0,0,0.00,0.00,None
1,00-0016919,A.Vinatieri,Adam Vinatieri,K,SPEC,https://static.www.nfl.com/image/private/f_aut...,2017,1,REG,IND,...,0,0.0,0,0,0,0,0,0.00,0.00,None
2,00-0019596,T.Brady,Tom Brady,QB,QB,https://static.www.nfl.com/image/private/f_aut...,2017,1,REG,NE,...,0,NaN,0,0,0,0,0,10.68,10.68,None
3,00-0019714,S.Lechler,Shane Lechler,P,SPEC,https://static.www.nfl.com/image/private/f_aut...,2017,1,REG,HOU,...,0,NaN,0,0,0,0,0,0.00,0.00,None
4,00-0020531,D.Brees,Drew Brees,QB,QB,https://static.www.nfl.com/image/private/f_aut...,2017,1,REG,NO,...,0,NaN,0,0,0,0,0,15.64,15.64,None


#### Load Seasonal Data

In [5]:
season_data = nfl.load_player_stats(seasons=[2017,2018,2019,2020,2021,2022,2023,2024], summary_level="reg")

In [6]:
season_data = season_data.to_pandas()

In [7]:
season_data.head()

,player_id,player_name,player_display_name,position,position_group,headshot_url,season,season_type,recent_team,games,...,pat_missed,pat_blocked,pat_pct,gwfg_made,gwfg_att,gwfg_missed,gwfg_blocked,gwfg_distance_list,fantasy_points,fantasy_points_ppr
0,00-0004091,P.Dawson,Phil Dawson,K,SPEC,https://static.www.nfl.com/image/private/f_aut...,2017,REG,ARI,16,...,1,2,0.884615,1,1,0,0,30,0.00,0.00
1,00-0016919,A.Vinatieri,Adam Vinatieri,K,SPEC,https://static.www.nfl.com/image/private/f_aut...,2017,REG,IND,15,...,2,0,0.916667,1,1,0,0,51,0.00,0.00
2,00-0019596,T.Brady,Tom Brady,QB,QB,https://static.www.nfl.com/image/private/f_aut...,2017,REG,NE,16,...,0,0,NaN,0,0,0,0,None,295.88,295.88
3,00-0019714,S.Lechler,Shane Lechler,P,SPEC,https://static.www.nfl.com/image/private/f_aut...,2017,REG,HOU,16,...,0,0,NaN,0,0,0,0,None,0.00,0.00
4,00-0020531,D.Brees,Drew Brees,QB,QB,https://static.www.nfl.com/image/private/f_aut...,2017,REG,NO,16,...,0,0,NaN,0,0,0,0,None,262.56,262.56


In [8]:
season_data.columns.to_list()

['player_id',
 'player_name',
 'player_display_name',
 'position',
 'position_group',
 'headshot_url',
 'season',
 'season_type',
 'recent_team',
 'games',
 'completions',
 'attempts',
 'passing_yards',
 'passing_tds',
 'passing_interceptions',
 'sacks_suffered',
 'sack_yards_lost',
 'sack_fumbles',
 'sack_fumbles_lost',
 'passing_air_yards',
 'passing_yards_after_catch',
 'passing_first_downs',
 'passing_epa',
 'passing_cpoe',
 'passing_2pt_conversions',
 'pacr',
 'carries',
 'rushing_yards',
 'rushing_tds',
 'rushing_fumbles',
 'rushing_fumbles_lost',
 'rushing_first_downs',
 'rushing_epa',
 'rushing_2pt_conversions',
 'receptions',
 'targets',
 'receiving_yards',
 'receiving_tds',
 'receiving_fumbles',
 'receiving_fumbles_lost',
 'receiving_air_yards',
 'receiving_yards_after_catch',
 'receiving_first_downs',
 'receiving_epa',
 'receiving_2pt_conversions',
 'racr',
 'target_share',
 'air_yards_share',
 'wopr',
 'special_teams_tds',
 'def_tackles_solo',
 'def_tackles_with_assist',
 '

In [9]:
season_data = season_data.merge(weekly_stats[['player_id', 'position']], on='player_id', how='left')
season_data = season_data.drop('position_y', axis=1)
season_data = season_data.rename(columns={'position_x': 'position'})

#### QB Data

In [10]:
qb_data = weekly_stats[weekly_stats['position'] == 'QB']
qb_data = qb_data[qb_data['season_type'] == 'REG']

In [11]:
season_data.head()

,player_id,player_name,player_display_name,position,position_group,headshot_url,season,season_type,recent_team,games,...,pat_missed,pat_blocked,pat_pct,gwfg_made,gwfg_att,gwfg_missed,gwfg_blocked,gwfg_distance_list,fantasy_points,fantasy_points_ppr
0,00-0004091,P.Dawson,Phil Dawson,K,SPEC,https://static.www.nfl.com/image/private/f_aut...,2017,REG,ARI,16,...,1,2,0.884615,1,1,0,0,30,0.0,0.0
1,00-0004091,P.Dawson,Phil Dawson,K,SPEC,https://static.www.nfl.com/image/private/f_aut...,2017,REG,ARI,16,...,1,2,0.884615,1,1,0,0,30,0.0,0.0
2,00-0004091,P.Dawson,Phil Dawson,K,SPEC,https://static.www.nfl.com/image/private/f_aut...,2017,REG,ARI,16,...,1,2,0.884615,1,1,0,0,30,0.0,0.0
3,00-0004091,P.Dawson,Phil Dawson,K,SPEC,https://static.www.nfl.com/image/private/f_aut...,2017,REG,ARI,16,...,1,2,0.884615,1,1,0,0,30,0.0,0.0
4,00-0004091,P.Dawson,Phil Dawson,K,SPEC,https://static.www.nfl.com/image/private/f_aut...,2017,REG,ARI,16,...,1,2,0.884615,1,1,0,0,30,0.0,0.0


In [12]:
seasonal_qb_data = season_data[(season_data['position'] == 'QB') & ((season_data['attempts'] >= 50) | (season_data['completions'] >= 30) | (season_data['carries'] >= 20))]

In [13]:
seasonal_qb_data = seasonal_qb_data[['player_id']].drop_duplicates()

In [14]:
qb_data = qb_data.merge(seasonal_qb_data, on = 'player_id', how = 'inner')

#### RB Data

In [15]:
rb_data = weekly_stats[weekly_stats['position'] == 'RB']

In [16]:
seasonal_rb_data = season_data[(season_data['position'] == 'RB') & ((season_data['carries'] >= 15) | (season_data['targets'] >= 10) | (season_data['target_share'] >= 0.025))]

In [17]:
seasonal_rb_data = seasonal_rb_data[['player_id']].drop_duplicates()

In [18]:
rb_data = rb_data.merge(seasonal_rb_data, on = 'player_id', how = 'inner')

#### WR/TE Data

In [19]:
wrte_data = weekly_stats[weekly_stats['position'].isin(['WR', 'TE'])].copy()

In [ ]:
seasonal_wrte = season_data[(season_data['position'].isin(['WR', 'TE'])) & ((season_data['targets'] >= 20) | (season_data['receptions'] >= 10) | (season_data['target_share'] >= 0.05) | (season_data['games'] >= 1))][['player_id']].drop_duplicates()

In [21]:
wrte_data = wrte_data.merge(seasonal_wrte, on=['player_id'], how='inner')

###### In the code above, we did some minor data cleaning to ensure that the model was working with players that have actually played and accumulated a significant number of statistics according to their respective position. The thresholds we set were arbitrary and we thought they made sense to cut down the number of players the model had to work with. The reason we did this is because for people who own fantasy football teams, it is not of much value for the model to output a prediction for players who may be on the practice squad or do not play a significant number of snaps.

#### Data Cleaning

In [22]:
qb_data = qb_data[['player_id', 'player_name', 'team', 'season', 'week',
       'season_type', 'opponent_team', 'completions', 'attempts',
       'passing_yards', 'passing_tds', 'passing_interceptions', 'sacks_suffered',
       'sack_fumbles', 'sack_fumbles_lost', 'passing_air_yards', 'carries', 'rushing_yards',
       'rushing_tds', 'rushing_fumbles', 'rushing_fumbles_lost','fantasy_points_ppr']]

In [23]:
rb_data = rb_data[['player_id', 'player_name', 'team', 'season', 'week',
       'season_type', 'opponent_team', 'carries', 'rushing_yards',
       'rushing_tds', 'rushing_fumbles', 'rushing_fumbles_lost', 'targets', 'receptions', 'receiving_yards',
       'receiving_tds', 'receiving_fumbles', 'receiving_fumbles_lost','fantasy_points_ppr']]

In [24]:
wrte_data = wrte_data[['player_id', 'player_name', 'team', 'season', 'week',
       'season_type', 'opponent_team', 'targets', 'receptions', 'receiving_yards',
       'receiving_tds', 'receiving_fumbles', 'receiving_fumbles_lost','fantasy_points_ppr']]

#### Function for Rolling Weekly Statistics

###### This function generates  rolling and lag-based features from weekly player statistics. It takes a dataframe containing player performance data and creates new columns representing rolling average statistics over the previous week, 3 weeks, and 5 weeks. The idea behind this function is that if a player is doing well over the past few weeks, the model can pick up on those signals and identify a high performing player.

In [ ]:
def add_rolling_weekly_features(weekly_data: pd.DataFrame, stat_cols: List[str],) -> pd.DataFrame:

    df = weekly_data.copy()
    df = df.sort_values(["player_id", "season", "week"]).reset_index(drop=True)

    
    for col in stat_cols:
        df[col] = pd.to_numeric(df[col], errors="coerce").fillna(0.0)

    grouped = df.groupby(["player_id", "season"])

    for col in stat_cols:
        
        df[f"{col}_lag1"] = grouped[col].shift(1)

        
        for window in (3,5):
            df[f"{col}_roll{window}_mean"] = grouped[col].transform(
                lambda s: s.shift(1).rolling(window, min_periods=1).mean()
            )

    return df

#### Function to Add Previous Season's Statistics

###### This function generates previous season performance features for each player and adds them to the weekly dataset. The goal is to give the model historical context about a player’s past performance. The idea is that the model should know if a player is typically a high performer or low performer. This means that all players won't be treated the same, especially during the first few weeks where the model doesn't have much information about the player's current season performance.

In [ ]:
def add_prev_season_features(weekly_data: pd.DataFrame, seasonal_data: pd.DataFrame, prev_cols: List[str],) -> pd.DataFrame:
    

    df = weekly_data.copy()
    prev = seasonal_data[['player_id', 'season'] + prev_cols].copy()
  
    prev['season'] = prev['season'] + 1

    prev = prev.rename(columns={c: f"{c}_prev" for c in prev_cols})

    
    df = df.merge(prev, on=['player_id', 'season'], how="left")

    #Rookie players 
    for c in prev_cols:
        df[f"{c}_prev"] = df[f"{c}_prev"].fillna(0)

    return df


#### Creating Rolling Data for QBs

In [ ]:
seasonal_qb_data = season_data[(season_data["position"] == "QB") & ((season_data["attempts"] >= 50) |(season_data["completions"] >= 30) | (season_data["carries"] >= 20)) &    (season_data["season_type"] == "REG")].copy()

seasonal_qb_data = seasonal_qb_data.drop_duplicates(subset=["player_id","season"])
print("seasonal_qb_data rows:", seasonal_qb_data.shape[0])

seasonal_qb_data rows: 421


In [28]:
qb_data.sort_values(by=['player_id', 'season', 'week'], inplace=True)

In [29]:
qb_data.columns

Index(['player_id', 'player_name', 'team', 'season', 'week', 'season_type',
       'opponent_team', 'completions', 'attempts', 'passing_yards',
       'passing_tds', 'passing_interceptions', 'sacks_suffered',
       'sack_fumbles', 'sack_fumbles_lost', 'passing_air_yards', 'carries',
       'rushing_yards', 'rushing_tds', 'rushing_fumbles',
       'rushing_fumbles_lost', 'fantasy_points_ppr'],
      dtype='object')

In [30]:
seasonal_qb_data.columns

Index(['player_id', 'player_name', 'player_display_name', 'position',
       'position_group', 'headshot_url', 'season', 'season_type',
       'recent_team', 'games',
       ...
       'pat_missed', 'pat_blocked', 'pat_pct', 'gwfg_made', 'gwfg_att',
       'gwfg_missed', 'gwfg_blocked', 'gwfg_distance_list', 'fantasy_points',
       'fantasy_points_ppr'],
      dtype='object', length=113)

In [ ]:
QB_WEEKLY_STATS = ["attempts", "completions", "passing_yards", "passing_tds", "passing_interceptions", "sacks_suffered", "passing_air_yards", "carries", "rushing_yards", "rushing_tds", "rushing_fumbles_lost", "sack_fumbles_lost"]

In [ ]:
QB_PREV_SEASON_COLS = ["attempts", "completions", "passing_yards", "passing_tds", "passing_interceptions", "sacks_suffered", "passing_air_yards","carries", "rushing_yards", "rushing_tds", "rushing_fumbles_lost", "sack_fumbles_lost","fantasy_points_ppr", "games"]

In [ ]:
qb_data = add_prev_season_features(weekly_data=qb_data, seasonal_data=seasonal_qb_data, prev_cols=QB_PREV_SEASON_COLS)

In [ ]:
qb_data = add_rolling_weekly_features(weekly_data=qb_data,stat_cols=QB_WEEKLY_STATS)

#### Rolling RB Data

In [ ]:
seasonal_rb_data = season_data[(season_data['position'] == 'RB') & ((season_data['carries'] >= 15) | (season_data['targets'] >= 10) | (season_data['target_share'] >= 0.025)) & (season_data["season_type"] == "REG")].copy()
seasonal_rb_data = seasonal_rb_data.drop_duplicates(subset=["player_id","season"])

In [36]:
seasonal_rb_data.columns

Index(['player_id', 'player_name', 'player_display_name', 'position',
       'position_group', 'headshot_url', 'season', 'season_type',
       'recent_team', 'games',
       ...
       'pat_missed', 'pat_blocked', 'pat_pct', 'gwfg_made', 'gwfg_att',
       'gwfg_missed', 'gwfg_blocked', 'gwfg_distance_list', 'fantasy_points',
       'fantasy_points_ppr'],
      dtype='object', length=113)

In [ ]:
RB_PREV_SEASON_COLS = ['carries', 'rushing_yards','rushing_tds', 'rushing_fumbles', 'rushing_fumbles_lost', 'targets', 'receptions', 'receiving_yards','receiving_tds', 'receiving_fumbles', 'receiving_fumbles_lost', 'fantasy_points_ppr', 'games']

In [ ]:
RB_WEEKLY_STATS = ['carries', 'rushing_yards','rushing_tds', 'rushing_fumbles', 'rushing_fumbles_lost', 'targets', 'receptions', 'receiving_yards', 'receiving_tds', 'receiving_fumbles','receiving_fumbles_lost']

In [ ]:
rb_data = add_prev_season_features(weekly_data=rb_data, seasonal_data=seasonal_rb_data, prev_cols=RB_PREV_SEASON_COLS)

In [ ]:
rb_data = add_rolling_weekly_features(weekly_data=rb_data, stat_cols=RB_WEEKLY_STATS)

In [41]:
rb_data.columns

Index(['player_id', 'player_name', 'team', 'season', 'week', 'season_type',
       'opponent_team', 'carries', 'rushing_yards', 'rushing_tds',
       'rushing_fumbles', 'rushing_fumbles_lost', 'targets', 'receptions',
       'receiving_yards', 'receiving_tds', 'receiving_fumbles',
       'receiving_fumbles_lost', 'fantasy_points_ppr', 'carries_prev',
       'rushing_yards_prev', 'rushing_tds_prev', 'rushing_fumbles_prev',
       'rushing_fumbles_lost_prev', 'targets_prev', 'receptions_prev',
       'receiving_yards_prev', 'receiving_tds_prev', 'receiving_fumbles_prev',
       'receiving_fumbles_lost_prev', 'fantasy_points_ppr_prev', 'games_prev',
       'carries_lag1', 'carries_roll3_mean', 'carries_roll5_mean',
       'rushing_yards_lag1', 'rushing_yards_roll3_mean',
       'rushing_yards_roll5_mean', 'rushing_tds_lag1',
       'rushing_tds_roll3_mean', 'rushing_tds_roll5_mean',
       'rushing_fumbles_lag1', 'rushing_fumbles_roll3_mean',
       'rushing_fumbles_roll5_mean', 'rushing_

#### Rolling WR/TE Data

In [ ]:
WRTE_PREV_SEASON_COLS = [ 'targets', 'receptions', 'receiving_yards', 'receiving_tds', 'receiving_fumbles', 'receiving_fumbles_lost', 'fantasy_points_ppr', 'games']

In [ ]:
WRTE_WEEKLY_STATS = ['targets','receptions', 'receiving_yards', 'receiving_tds', 'receiving_fumbles','receiving_fumbles_lost']

In [ ]:
seasonal_wrte = season_data[(season_data['position'].isin(['WR', 'TE'])) & ((season_data['targets'] >= 20) | (season_data['receptions'] >= 10) | (season_data['target_share'] >= 0.05) | (season_data['games'] >= 1)) & (season_data["season_type"] == "REG")].copy()
seasonal_wrte_data = seasonal_wrte.drop_duplicates(subset=["player_id","season"])

In [45]:
wrte_data = wrte_data[wrte_data['season_type'] == 'REG']

In [ ]:
wrte_data = add_prev_season_features(weekly_data=wrte_data, seasonal_data=seasonal_wrte_data, prev_cols=WRTE_PREV_SEASON_COLS)

In [ ]:
wrte_data = add_rolling_weekly_features(weekly_data=wrte_data, stat_cols=WRTE_WEEKLY_STATS)

In [48]:
wrte_data.columns

Index(['player_id', 'player_name', 'team', 'season', 'week', 'season_type',
       'opponent_team', 'targets', 'receptions', 'receiving_yards',
       'receiving_tds', 'receiving_fumbles', 'receiving_fumbles_lost',
       'fantasy_points_ppr', 'targets_prev', 'receptions_prev',
       'receiving_yards_prev', 'receiving_tds_prev', 'receiving_fumbles_prev',
       'receiving_fumbles_lost_prev', 'fantasy_points_ppr_prev', 'games_prev',
       'targets_lag1', 'targets_roll3_mean', 'targets_roll5_mean',
       'receptions_lag1', 'receptions_roll3_mean', 'receptions_roll5_mean',
       'receiving_yards_lag1', 'receiving_yards_roll3_mean',
       'receiving_yards_roll5_mean', 'receiving_tds_lag1',
       'receiving_tds_roll3_mean', 'receiving_tds_roll5_mean',
       'receiving_fumbles_lag1', 'receiving_fumbles_roll3_mean',
       'receiving_fumbles_roll5_mean', 'receiving_fumbles_lost_lag1',
       'receiving_fumbles_lost_roll3_mean',
       'receiving_fumbles_lost_roll5_mean'],
      dtype='

#### Building Defensive Data

###### The next few functions allow us to create features to give the model signals on how good the defense the player is facing is. A player playing a good defense may underperform. These features generated by the fuctions allow the model to learn based on defensive signals.

In [ ]:
def build_defense_weekly(weekly_data: pd.DataFrame,is_qb: bool = False, is_rb: bool = False,) -> pd.DataFrame:
    
    defense = weekly_data.copy()
    
    if is_qb:
        rename_cols = {
            "opponent_team": "defense_team",
            "passing_yards": "pass_yards_allowed",
            "passing_tds": "pass_tds_allowed",
            "fantasy_points_ppr": "qb_fp_allowed"
        }
        keep_cols = [
            "defense_team", "season", "week",
            "pass_yards_allowed", "pass_tds_allowed", "qb_fp_allowed"
        ]
    elif is_rb:
        rename_cols = {
            "opponent_team": "defense_team",
            "rushing_yards": "rb_rush_yards_allowed",
            "rushing_tds": "rb_rush_tds_allowed",
            "targets": "rb_targets_allowed",
            "receptions": "rb_receptions_allowed",
            "receiving_yards": "rb_rec_yards_allowed",
            "receiving_tds": "rb_rec_tds_allowed",
            "fantasy_points_ppr": "rb_fp_allowed",
        }
        keep_cols = [
            "defense_team", "season", "week",
            "rb_rush_yards_allowed", "rb_rush_tds_allowed",
            "rb_targets_allowed", "rb_receptions_allowed",
            "rb_rec_yards_allowed", "rb_rec_tds_allowed", "rb_fp_allowed"
        ]
    else:  
        rename_cols = {
            "opponent_team": "defense_team",
            "targets": "wrte_targets_allowed",
            "receptions": "wrte_receptions_allowed",
            "receiving_yards": "wrte_rec_yards_allowed",
            "receiving_tds": "wrte_rec_tds_allowed",
            "fantasy_points_ppr": "wrte_fp_allowed",
        }
        keep_cols = [
            "defense_team", "season", "week",
            "wrte_targets_allowed", "wrte_receptions_allowed",
            "wrte_rec_yards_allowed", "wrte_rec_tds_allowed", "wrte_fp_allowed"
        ]
    
    defense = defense.rename(columns=rename_cols)
    return defense[keep_cols]

In [ ]:
def aggregate_defense_weekly(def_df: pd.DataFrame) -> pd.DataFrame:
    num_cols = [c for c in def_df.columns if c not in ["defense_team", "season", "week"]]
    return (def_df.groupby(["defense_team", "season", "week"], as_index=False)[num_cols].sum())

In [ ]:
def add_defense_rolling_features(defense_weekly: pd.DataFrame, stat_cols: list[str] = None) -> pd.DataFrame:
  
    df = defense_weekly.sort_values(["defense_team", "season", "week"]).copy()
    g = df.groupby(["defense_team", "season"], sort=False)
    
    if stat_cols is None:
        stat_cols = [c for c in df.columns if c not in ["defense_team", "season", "week"]]
    
    for stat in stat_cols:
        prev = g[stat].shift(1)
        for window in (3,5):
            df[f"{stat}_roll{window}_mean"] = prev.rolling(window, min_periods=1).mean()
    
    return df

In [ ]:
def merge_defense_into_weekly(weekly_df: pd.DataFrame, defense_features: pd.DataFrame) -> pd.DataFrame:

    return (weekly_df.merge(defense_features, left_on=["opponent_team", "season", "week"], right_on=["defense_team", "season", "week"], how="left").drop(columns=["defense_team"]))

In [ ]:
qb_def = build_defense_weekly(qb_data, is_qb=True)
qb_def = aggregate_defense_weekly(qb_def)
qb_def = add_defense_rolling_features(qb_def)
qb_data = merge_defense_into_weekly(qb_data, qb_def)

In [54]:
qb_data.head()

,player_id,player_name,team,season,week,season_type,opponent_team,completions,attempts,passing_yards,...,sack_fumbles_lost_roll5_mean,pass_yards_allowed,pass_tds_allowed,qb_fp_allowed,pass_yards_allowed_roll3_mean,pass_yards_allowed_roll5_mean,pass_tds_allowed_roll3_mean,pass_tds_allowed_roll5_mean,qb_fp_allowed_roll3_mean,qb_fp_allowed_roll5_mean
0,00-0019596,T.Brady,NE,2017,1,REG,KC,16,36,267,...,NaN,267,0,10.68,225.000000,226.75,0.5,1.00,11.750000,15.145
1,00-0019596,T.Brady,NE,2017,2,REG,NO,30,39,447,...,0.000000,447,3,30.78,313.500000,251.25,3.0,1.75,25.390000,18.000
2,00-0019596,T.Brady,NE,2017,3,REG,HOU,25,35,378,...,0.000000,378,5,35.72,174.500000,219.75,0.5,1.00,9.880000,12.365
3,00-0019596,T.Brady,NE,2017,4,REG,CAR,32,45,307,...,0.333333,307,2,20.48,179.333333,183.25,1.0,1.25,11.506667,12.230
4,00-0019596,T.Brady,NE,2017,5,REG,TB,30,40,303,...,0.250000,303,1,12.62,319.333333,315.25,2.0,2.25,22.106667,23.035


#### Building Defensive Data for RBs

In [ ]:
rb_def = build_defense_weekly(rb_data, is_rb=True)
rb_def = aggregate_defense_weekly(rb_def)
rb_def = add_defense_rolling_features(rb_def)
rb_data = merge_defense_into_weekly(rb_data, rb_def)

In [56]:
rb_data.head()

,player_id,player_name,team,season,week,season_type,opponent_team,carries,rushing_yards,rushing_tds,...,rb_targets_allowed_roll3_mean,rb_targets_allowed_roll5_mean,rb_receptions_allowed_roll3_mean,rb_receptions_allowed_roll5_mean,rb_rec_yards_allowed_roll3_mean,rb_rec_yards_allowed_roll5_mean,rb_rec_tds_allowed_roll3_mean,rb_rec_tds_allowed_roll5_mean,rb_fp_allowed_roll3_mean,rb_fp_allowed_roll5_mean
0,00-0023500,F.Gore,IND,2017,1,REG,LA,10,42,0,...,4.500000,6.25,3.500000,4.75,30.500000,42.00,0.000000,0.00,26.300000,26.525
1,00-0023500,F.Gore,IND,2017,2,REG,ARI,14,46,1,...,11.000000,11.00,9.000000,9.00,38.000000,38.00,1.000000,1.00,23.900000,23.900
2,00-0023500,F.Gore,IND,2017,3,REG,CLE,25,57,1,...,7.000000,6.50,5.000000,5.50,36.500000,33.75,0.500000,0.25,22.300000,21.500
3,00-0023500,F.Gore,IND,2017,4,REG,SEA,12,46,0,...,3.666667,4.75,2.000000,3.25,12.333333,12.75,0.000000,0.00,16.466667,16.175
4,00-0023500,F.Gore,IND,2017,5,REG,SF,14,48,0,...,11.000000,10.50,7.333333,7.25,64.000000,61.75,0.333333,0.50,28.366667,28.200


#### Building Defensive Data for WR/TE

In [ ]:
wrte_def = build_defense_weekly(wrte_data, is_wrte=True)
wrte_def = aggregate_defense_weekly(wrte_def)
wrte_def = add_defense_rolling_features(wrte_def)
wrte_data = merge_defense_into_weekly(wrte_data, wrte_def)

In [58]:
wrte_data.columns

Index(['player_id', 'player_name', 'team', 'season', 'week', 'season_type',
       'opponent_team', 'targets', 'receptions', 'receiving_yards',
       'receiving_tds', 'receiving_fumbles', 'receiving_fumbles_lost',
       'fantasy_points_ppr', 'targets_prev', 'receptions_prev',
       'receiving_yards_prev', 'receiving_tds_prev', 'receiving_fumbles_prev',
       'receiving_fumbles_lost_prev', 'fantasy_points_ppr_prev', 'games_prev',
       'targets_lag1', 'targets_roll3_mean', 'targets_roll5_mean',
       'receptions_lag1', 'receptions_roll3_mean', 'receptions_roll5_mean',
       'receiving_yards_lag1', 'receiving_yards_roll3_mean',
       'receiving_yards_roll5_mean', 'receiving_tds_lag1',
       'receiving_tds_roll3_mean', 'receiving_tds_roll5_mean',
       'receiving_fumbles_lag1', 'receiving_fumbles_roll3_mean',
       'receiving_fumbles_roll5_mean', 'receiving_fumbles_lost_lag1',
       'receiving_fumbles_lost_roll3_mean',
       'receiving_fumbles_lost_roll5_mean', 'wrte_targets_

In [ ]:
qb_data = qb_data.drop(columns=['season_type', 'completions', 'attempts','passing_yards', 'passing_tds', 'passing_interceptions', 'sacks_suffered', 'sack_fumbles', 'sack_fumbles_lost', 'passing_air_yards', 'carries','rushing_yards', 'rushing_tds', 'rushing_fumbles','rushing_fumbles_lost', 'pass_yards_allowed', 'pass_tds_allowed', 'qb_fp_allowed'])

In [ ]:
rb_data = rb_data.drop(columns=['season_type', 'carries', 'rushing_yards', 'rushing_tds', 'rushing_fumbles', 'rushing_fumbles_lost', 'targets', 'receptions', 'receiving_yards', 'receiving_tds', 'receiving_fumbles', 'receiving_fumbles_lost', 'rb_rush_yards_allowed', 'rb_rush_tds_allowed', 'rb_targets_allowed', 'rb_receptions_allowed','rb_rec_yards_allowed', 'rb_rec_tds_allowed', 'rb_fp_allowed'])

In [ ]:
wrte_data = wrte_data.drop(columns=['season_type', 'targets', 'receptions', 'receiving_yards', 'receiving_tds', 'receiving_fumbles','receiving_fumbles_lost', 'wrte_targets_allowed','wrte_receptions_allowed', 'wrte_rec_yards_allowed','wrte_rec_tds_allowed', 'wrte_fp_allowed'])

In [62]:
wrte_data = wrte_data.sort_values(by=['player_id', 'season', 'week'])
qb_data = qb_data.sort_values(by=['player_id', 'season', 'week'])
rb_data = rb_data.sort_values(by=['player_id', 'season', 'week'])

###### We will use PostgreSQL via Supabase to store the data for QBs, RBS, WRs, and TEs. This will make the information easy to query when building the training, validation, and testing sets for the models.|

In [63]:
# Load environment variables from .env
load_dotenv()

# Fetch variables
USER = os.getenv("user")
PASSWORD = os.getenv("password")
HOST = os.getenv("host")
PORT = os.getenv("port")
DBNAME = os.getenv("dbname")

# Construct the SQLAlchemy connection string
DATABASE_URL = f"postgresql+psycopg2://{USER}:{PASSWORD}@{HOST}:{PORT}/{DBNAME}?sslmode=require"

# Create the SQLAlchemy engine
engine = create_engine(DATABASE_URL)
# If using Transaction Pooler or Session Pooler, we want to ensure we disable SQLAlchemy client side pooling -
# https://docs.sqlalchemy.org/en/20/core/pooling.html#switching-pool-implementations
# engine = create_engine(DATABASE_URL, poolclass=NullPool)

# Test the connection
try:
    with engine.connect() as connection:
        print("Connection successful!")
except Exception as e:
    print(f"Failed to connect: {e}")

Connection successful!


In [64]:
qb_data.to_sql("QBPointProjection", engine, if_exists="replace", index=False)
rb_data.to_sql("RBPointProjection", engine, if_exists="replace", index=False)
wrte_data.to_sql("WRTEPointProjection", engine, if_exists="replace", index=False)

85